In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import lightgbm as lgb

In [4]:
df= pd.read_csv("historical_data.csv")
df.head()

,market_id,created_at,actual_delivery_time,store_id,store_primary_category,order_protocol,total_items,subtotal,num_distinct_items,min_item_price,max_item_price,total_onshift_dashers,total_busy_dashers,total_outstanding_orders,estimated_order_place_duration,estimated_store_to_consumer_driving_duration
0,1.0,2015-02-06 22:24:17,2015-02-06 23:27:16,1845,american,1.0,4,3441,4,557,1239,33.0,14.0,21.0,446,861.0
1,2.0,2015-02-10 21:49:25,2015-02-10 22:56:29,5477,mexican,2.0,1,1900,1,1400,1400,1.0,2.0,2.0,446,690.0
2,3.0,2015-01-22 20:39:28,2015-01-22 21:09:09,5477,NaN,1.0,1,1900,1,1900,1900,1.0,0.0,0.0,446,690.0
3,3.0,2015-02-03 21:21:45,2015-02-03 22:13:00,5477,NaN,1.0,6,6900,5,600,1800,1.0,1.0,2.0,446,289.0
4,3.0,2015-02-15 02:40:36,2015-02-15 03:20:26,5477,NaN,1.0,3,3900,3,1100,1600,6.0,6.0,9.0,446,650.0


In [5]:
# 2. Convert text timestamps into real computer clocks
df['created_at'] = pd.to_datetime(df['created_at'])
df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'])

# 3. Calculate total delivery seconds (End Time minus Start Time)
df['total_delivery_seconds'] = (df['actual_delivery_time'] - df['created_at']).dt.total_seconds()

# Peek at our new target column
df[['created_at', 'actual_delivery_time', 'total_delivery_seconds']].head()

,created_at,actual_delivery_time,total_delivery_seconds
0,2015-02-06 22:24:17,2015-02-06 23:27:16,3779.0
1,2015-02-10 21:49:25,2015-02-10 22:56:29,4024.0
2,2015-01-22 20:39:28,2015-01-22 21:09:09,1781.0
3,2015-02-03 21:21:45,2015-02-03 22:13:00,3075.0
4,2015-02-15 02:40:36,2015-02-15 03:20:26,2390.0


In [6]:
# 1. Drop rows where delivery time is missing or completely impossible (gliches)
df = df.dropna(subset=['total_delivery_seconds'])
df = df[df['total_delivery_seconds'] > 0]

# 2. Fill empty restaurant food categories with 'unknown'
df['store_primary_category'] = df['store_primary_category'].fillna('unknown')

# 3. Print out the number of rows left to make sure everything looks good
print(f"🧹 Clean up complete! We have {df.shape[0]} real deliveries ready for our machine.")

🧹 Clean up complete! We have 197421 real deliveries ready for our machine.


In [7]:
# Super Clue 1: Find how many drivers are totally free
df['free_dashers'] = df['total_onshift_dashers'] - df['total_busy_dashers']

# Super Clue 2: Extract the hour of the day from the order time
df['order_hour'] = df['created_at'].dt.hour

# Super Clue 3: Sum the simple estimated time stages together
df['estimated_total_baseline'] = df['estimated_order_place_duration'] + df['estimated_store_to_consumer_driving_duration']

# Let's peek at our awesome new clues!
df[['total_onshift_dashers', 'total_busy_dashers', 'free_dashers', 'order_hour', 'estimated_total_baseline']].head()

,total_onshift_dashers,total_busy_dashers,free_dashers,order_hour,estimated_total_baseline
0,33.0,14.0,19.0,22,1307.0
1,1.0,2.0,-1.0,21,1136.0
2,1.0,0.0,1.0,20,1136.0
3,1.0,1.0,0.0,21,735.0
4,6.0,6.0,0.0,2,1096.0


In [8]:
# 1. Select the specific clues we want to use
clue_columns = [
    'market_id', 'store_id', 'order_protocol', 'total_items', 'subtotal', 
    'num_distinct_items', 'min_item_price', 'max_item_price', 
    'total_onshift_dashers', 'total_busy_dashers', 'total_outstanding_orders',
    'free_dashers', 'order_hour', 'estimated_total_baseline', 'store_primary_category'
]

X = df[clue_columns]
y = df['total_delivery_seconds']

# 2. Turn the text categories (like pizza, sushi) into numbers automatically
X = pd.get_dummies(X, columns=['store_primary_category', 'market_id', 'order_protocol', 'store_id'], drop_first=True)

print("🎯 Clues (X) and Answers (y) are officially separated and converted to numbers!")

🎯 Clues (X) and Answers (y) are officially separated and converted to numbers!


In [9]:
# Split the data into training (study) and testing (quiz) sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"📚 Study Pile (Training): {X_train.shape[0]} deliveries")
print(f"📝 Quiz Pile (Testing): {X_test.shape[0]} deliveries")

📚 Study Pile (Training): 157936 deliveries
📝 Quiz Pile (Testing): 39485 deliveries


In [10]:
# 1. Create our LightGBM regression model object
model = lgb.LGBMRegressor(random_state=42, n_estimators=100, learning_rate=0.1)

# 2. Tell the brain to start studying! (This might take a few seconds)
model.fit(X_train, y_train)

print("🎉 The Magic Brain has finished studying and is ready for the quiz!")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001470 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 5896
[LightGBM] [Info] Number of data points in the train set: 157936, number of used features: 2053
[LightGBM] [Info] Start training from score 2917.174090
🎉 The Magic Brain has finished studying and is ready for the quiz!


In [11]:
# 1. Ask the model to make its guesses for the quiz pile
guesses = model.predict(X_test)

# 2. Calculate the average mistake in seconds
mae_seconds = mean_absolute_error(y_test, guesses)

# 3. Convert those seconds into normal minutes so humans can understand it!
mae_minutes = mae_seconds / 60

print(f"🏆 Final Report Card Score (MAE): {mae_minutes:.2f} minutes")

🏆 Final Report Card Score (MAE): 13.71 minutes
